# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ma5029blp-wq/ML-flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!pip -q install duckdb huggingface_hub
from google.colab import userdata

HF_TOKEN = userdata.get("internship")
print("Token loaded successfully!" if HF_TOKEN else "Token not found.")

Token loaded successfully!


In [8]:
import duckdb

con = duckdb.connect()

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

One row in the raw performance table represents a client-content observation for a specific report date. The expected grain is one row per **report_date + client_hash_id + content_hash_id.** The warehouse covers dates from **2025-01-27 to 2026-06-30**. A grain verification query found** 6,390 **duplicate groups, so duplicate records must be handled before aggregation or modeling. For analysis, daily observations are aggregated into client-content level features over defined time windows.

In [9]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [10]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name}: {n:,} rows")

dim_clients: 104 rows
dim_content: 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily: 78,835,655 rows
fact_query_90d: 2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2025-01-27,2026-06-30,78835655


In [12]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS c
FROM {TABLES['fact_daily']}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING c > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c
0,2026-06-13,client_8ddc46da5414ffd8,content_1d06a2c99a935e49,2
1,2026-06-18,client_810019792c9b8efc,content_fa5355f17fb01178,2
2,2026-06-22,client_1a8bf67cad4ee525,content_6f83f0f309476a88,2
3,2026-06-20,client_86ebc2f12c01f586,content_821fedaa25538d5a,2
4,2026-06-25,client_1a730cb2640a1abf,content_4112607d223e829e,2


In [13]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
WHERE report_date = '2026-06-13'
AND client_hash_id = 'client_8ddc46da5414ffd8'
AND content_hash_id = 'content_1d06a2c99a935e49'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-13,client_8ddc46da5414ffd8,content_1d06a2c99a935e49,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
1,2026-06-13,client_8ddc46da5414ffd8,content_1d06a2c99a935e49,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06


In [14]:
con.sql(f"""
SELECT
    COUNT(*) AS duplicate_groups,
    SUM(c - 1) AS duplicate_rows
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING c > 1
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_groups,duplicate_rows
0,6390,6390.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature:
The fields used as inputs for analysis or modeling are:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- scroll_events
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

These fields represent information available from content performance history and data availability signals.

Label / Proxy:
- is_declining_label

The label represents whether a content item experienced a significant decline in performance over a defined comparison window. It is derived from future performance changes and is not used as a feature.

Context:
- client_hash_id
- content_hash_id
- report_date
- month

These fields identify the entity, allow grouping and joining, and define the time period. They are not used as model features.

Excluded:
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other

These fields are excluded because AI source-level traffic information may depend on the outcome period and can introduce leakage if used to predict future content performance changes.

**These detailed AI source fields are excluded because they are not required for the current content decline analysis and may represent downstream traffic outcomes rather than pre-prediction signals.**

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
DESCRIBE SELECT *
FROM {TABLES['fact_daily']}
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [17]:
cols = [
    'gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'ga4_sessions',
    'scroll_events'
]

query = f"""
SELECT
"""

for c in cols:
    query += f"""
AVG(CASE WHEN {c} IS NULL THEN 1.0 ELSE 0 END) AS {c}_missing,
"""

query = query.rstrip(",") + f"""
FROM {TABLES['fact_daily']}
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions_missing,gsc_clicks_missing,gsc_avg_position_missing,ga4_sessions_missing,scroll_events_missing
0,0.001243,0.001243,0.632527,0.375913,0.375913


In [18]:
con.sql(f"""
SELECT
    ga4_data_available,
    COUNT(*) AS rows,
    AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS ga4_sessions_missing_rate
FROM {TABLES['fact_daily']}
GROUP BY ga4_data_available
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows,ga4_sessions_missing_rate
0,False,46383873,0.0
1,<NA>,29635327,1.0
2,True,2816455,0.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The data contract claims were verified using SQL queries.

The warehouse contains 78,835,655 rows with daily performance data from 2025-01-27 to 2026-06-30.

The expected grain is one observation per report_date, client_hash_id, and content_hash_id. A duplicate check found 6,390 duplicate groups, so duplicate records should be handled before aggregation or modeling.

Missing value checks show that missingness is not random. gsc_impressions and gsc_clicks have very low missing rates (~ 0.1%), while gsc_avg_position has high missingness (~63%) and GA4-related fields have missingness around 38%. Missingness is related to data availability patterns.

Client-level window checks show that different clients have different history start dates, confirming that the panel is unbalanced. Time-based analysis should account for differences in available history.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# verify total rows and date window
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
""").df()

,total_rows,start_date,end_date
0,78835655,2025-01-27,2026-06-30


In [21]:
# Verify grain

con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM {TABLES['fact_daily']}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count
0,2026-06-15,client_a2eeb8899886adde,content_9ce5ba9ac4527940,2
1,2026-06-17,client_1a730cb2640a1abf,content_965b9031838c130f,2
2,2026-06-17,client_1a730cb2640a1abf,content_5c5b96cd4d1829d8,2
3,2026-06-14,client_aef6ffea193da149,content_608ff8abdfba1469,2
4,2026-06-18,client_def0955f7a377868,content_40c868cfcb49537c,2


In [23]:
#Verify missing values
con.sql(f"""
SELECT
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS gsc_impressions_missing,
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS gsc_clicks_missing,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS gsc_avg_position_missing,
    AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS ga4_sessions_missing,
    AVG(CASE WHEN scroll_events IS NULL THEN 1.0 ELSE 0 END) AS scroll_events_missing
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions_missing,gsc_clicks_missing,gsc_avg_position_missing,ga4_sessions_missing,scroll_events_missing
0,0.001243,0.001243,0.632527,0.375913,0.375913


In [24]:
# Verify history windows by client
con.sql(f"""
SELECT
    client_hash_id,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) AS rows
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id
ORDER BY first_date
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,first_date,last_date,rows
0,client_9958f0a7ae1df715,2025-01-27,2026-06-30,1470881
1,client_ff644d8251367cbb,2025-01-27,2026-06-30,1246888
2,client_73cda7b4e4f265ea,2025-02-11,2026-06-30,8708971
3,client_fef1a8f436438636,2025-03-11,2026-06-30,2779933
4,client_62f4a7e64f5e0096,2025-06-07,2026-06-30,5676451
5,client_b10cb2997d0c7c86,2025-06-18,2026-06-30,1050146
6,client_65de48885f4ef01b,2025-06-21,2026-06-30,3296200
7,client_c182d11e4862a37d,2025-06-21,2026-06-30,282410
8,client_3197e6291363b4db,2025-06-29,2026-06-30,2598532
9,client_625b6439094e23e4,2025-07-01,2026-06-30,7589247


Window verification:
Client history windows were checked using MIN(report_date) and MAX(report_date) per client. The results show that clients have different starting dates, meaning the panel is unbalanced. Any time-based feature creation should account for differences in available history rather than assuming all clients have the same observation period.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset cannot fully explain why content performance changes. It only contains observed search and analytics metrics, not external factors such as algorithm updates, competitors, seasonality, or business decisions.

The panel has unbalanced history because clients have different data start dates, so comparisons across clients may be affected by different amounts of available history.

Some early rows may only contain GSC information because GA4 tracking was not available for all clients from the beginning. Missing GA4 values should not always be interpreted as zero activity.

Time windows can overlap when creating features and labels. Features must be created only from information available before the prediction period to avoid leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.